## Constants

In [ ]:
VOCAB_SIZE = 4096
SPECIAL_TOKENS = ["<unk>", "<bos>", "<eos>", "<pad>"]
UNKOWN_TOKEN = "<unk>"

## Load Dataset

In [ ]:
from datasets import load_dataset

# Load the TinyStories dataset
print("Downloading TinyStories dataset...")
dataset = load_dataset("roneneldan/TinyStories")

# Let's inspect what we just downloaded
print(dataset)

# Print a sample story to see what it looks like
print("\n--- Sample Story ---")
print(dataset["train"][0]["text"])

## Generate vocabulary

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Instantiate the BPE model backbone
tokenizer = Tokenizer(BPE(unk_token=UNKOWN_TOKEN))

# Pre-tokenizer to split text into words by whitespace
tokenizer.pre_tokenizer = Whitespace()

# Setup the trainer
trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,                          
    special_tokens=SPECIAL_TOKENS
)

# Create a generator to stream the dataset in chunks
# This feeds the text to the tokenizer 10,000 stories at a time
def batch_iterator(batch_size=10_000):
    for i in range(0, len(dataset["train"]), batch_size):
        yield dataset["train"][i : i + batch_size]["text"]

# Train the tokenizer using the iterator instead of a text file
print("Training BPE tokenizer from dataset (this may take a minute)...")
tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)

# Save the trained tokenizer configuration to disk
tokenizer.save("tinystories_bpe.json")
print("BPE Tokenizer successfully trained and saved!")